# EcoPredict AI 🌱
### Vehicle CO₂ Emission Prediction using Simple Linear Regression

This notebook compares model performance before and after an exact-duplicate-removal preprocessing experiment. Run cells from top to bottom. Keep the original CSV unchanged.

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 2. Load the original dataset

Place `Fuel_Consumption.csv` in the notebook's working directory before running the notebook.

In [ ]:
df = pd.read_csv("Fuel_Consumption.csv")
print("Original dataset shape:", df.shape)
display(df.head())
print("Columns:", df.columns.tolist())

## 3. Data Preprocessing and Audit

### 3.1 Inspect data quality

In [ ]:
print("Rows and columns:", df.shape)
print("\nData types:\n", df.dtypes)
print("\nMissing values:\n", df.isnull().sum())
print("\nExact duplicate rows:", df.duplicated().sum())
print("\nStatistical summary:")
display(df.describe())

### 3.2 Check valid values and repeated pairs

Repeated exact pairs are not automatically errors: without vehicle identifiers, some could be different vehicles with identical measurements.

In [ ]:
print("Engine size <= 0:", (df["Engine size (L)"] <= 0).sum())
print("Negative CO2 emissions:", (df["CO2 emissions (g/km)"] < 0).sum())
pair_counts = df.groupby(["Engine size (L)", "CO2 emissions (g/km)"]).size().reset_index(name="row_count")
print("Unique engine-size / CO2 pairs:", len(pair_counts))
print("Pairs appearing more than once:", (pair_counts["row_count"] > 1).sum())
display(pair_counts.sort_values("row_count", ascending=False).head(10))

### 3.3 Create the preprocessed dataset

For this experiment, remove exact duplicate rows while preserving the original file and all distinct engine-size/CO₂ pairs. This does not assume every repeated record is erroneous.

In [ ]:
cleaned_df = df.drop_duplicates(keep="first").reset_index(drop=True)
cleaned_df.to_csv("Final_Data_Preprocessed.csv", index=False)
print("Original rows:", len(df))
print("Preprocessed rows:", len(cleaned_df))
print("Rows removed:", len(df) - len(cleaned_df))
print("Remaining exact duplicates:", cleaned_df.duplicated().sum())
print("Preprocessed missing values:\n", cleaned_df.isnull().sum())
display(cleaned_df.head())

## 4. Machine Learning Model

We use only **Simple Linear Regression**. Input feature: `Engine size (L)`. Target: `CO2 emissions (g/km)`. The same 70/30 split settings are used for each dataset, but the test records differ; see the evaluation caveat in the comparison section.

### 4.1 Algorithm — Simple Linear Regression

In [ ]:
def train_slr(data):
    X = data[["Engine size (L)"]]
    y = data["CO2 emissions (g/km)"]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.30, random_state=42
    )
    model = LinearRegression()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return model, X_test, y_test, y_pred

### 4.2 Import Dataset 1 — Original Data

In [ ]:
original_df = pd.read_csv("Fuel_Consumption.csv")
print("Original dataset shape:", original_df.shape)
display(original_df.head())
original_model, X_test_original, y_test_original, y_pred_original = train_slr(original_df)

### 4.3 Graph 1 — Original Dataset

**Reserved for Radhika.** Add a scatter plot of the original observations and the fitted Simple Linear Regression line. Label axes with Engine size (L) and CO₂ emissions (g/km).

### 4.4 Import Dataset 2 — Preprocessed Data

In [ ]:
cleaned_df = pd.read_csv("Final_Data_Preprocessed.csv")
print("Preprocessed dataset shape:", cleaned_df.shape)
display(cleaned_df.head())
cleaned_model, X_test_cleaned, y_test_cleaned, y_pred_cleaned = train_slr(cleaned_df)

### 4.5 Graph 2 — Preprocessed Dataset

**Reserved for Radhika.** Add a scatter plot of the preprocessed observations and the fitted Simple Linear Regression line. Label axes with Engine size (L) and CO₂ emissions (g/km).

### 4.6 Results and Comparison — Nandini

MAE and RMSE should be lower for smaller prediction errors. R² is a goodness-of-fit metric, **not a percentage accuracy score**.

In [ ]:
def show_results(name, y_test, y_pred, model):
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    print(name)
    print(f"Equation: CO2 = {model.coef_[0]:.2f} * Engine size + {model.intercept_:.2f}")
    print(f"MAE: {mae:.2f} g/km")
    print(f"RMSE: {rmse:.2f} g/km")
    print(f"R²: {r2:.3f}")
    print()

show_results("BEFORE PREPROCESSING — Original Data", y_test_original, y_pred_original, original_model)
show_results("AFTER PREPROCESSING — Exact Duplicates Removed", y_test_cleaned, y_pred_cleaned, cleaned_model)

### Interpretation note

The initial split-based results are preliminary because each model is evaluated on different test records. Before making a firm claim that preprocessing improved performance, compare both models on the same held-out examples. Exact repeated pairs may be valid observations, so describe this as an experiment in removing exact duplicates.